# 42M model: quality-first continuation
Run CPU preparation first; export and verify ready.json from Drive. Resume own 20k weights and optimizer; train to 100k with batch 128, peak LR .00015, 500 warmup steps. Save and score every 10k checkpoint once. Review 30k and 40k before spending the full remaining budget. T4 timing is deferred at the user's request.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'CPU preparation only'
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
import os,sys,hashlib,tarfile,subprocess
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-bootstrap-v1/source.tar')
b=p.read_bytes()
assert hashlib.sha256(b).hexdigest()=='5e7dfc8879d57823683d2c7bd993db797a3015e5bf54725dac3546552ceb8c3c'
work=Path('/content/full-42m-prepare-source');work.mkdir(exist_ok=False)
(work/'source.tar').write_bytes(b)
with tarfile.open(work/'source.tar') as z:z.extractall(work,filter='data')
assert (work/'source_commit.txt').read_text().strip()=='69829cbe4f76a54907f1306df19e34065e75a75a'
os.environ['USE_TF']='0';os.environ['USE_FLAX']='0'
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True,timeout=180)
subprocess.run([sys.executable,'-u','-m','homeworks.hw2.full_training'],cwd=work,check=True,timeout=900)


After CPU files pass validation and export, change to a BF16 GPU with at least 40 GiB. Mount Drive, then run only the next cell. The launcher has a 9,000-second process-group timeout and releases compute. Verify release in Colab UI.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=180000)
from pathlib import Path
import hashlib
p=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/full-42m-100k-v1/launch.py')
s=p.read_bytes()
import json
r=json.loads((p.parent/'ready.json').read_text())
assert r['cpu_verified'] and hashlib.sha256(s).hexdigest()==r['launcher_sha256']
exec(compile(s,str(p),'exec'))
